# Validation Class Analysis (Per-class Correct/Incorrect + Confusion Heatmap)

- 입력: 특정 run_dir의 config와 best.pth
- 동작: fold의 validation 인덱스를 불러 logits/예측을 생성 →
  - 클래스별 정답/오답 막대그래프 저장
  - (정답 x 예측) 히트맵 저장
- 산출: reports/summary/<timestamp>/ 아래 PNG 저장

In [1]:
import os, time, pickle, json, math, pathlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')

ROOT = Path.cwd()
run_dir = Path('outputs/runs/20251110-060511_tf_efficientnet_b7_ns_448px_rot90_jitter_affine_rrc_mixup_only_ls0.05_cosine')  # 필요 시 교체
cfg = yaml.safe_load(open(run_dir/'config.yaml'))
fold_idx = int(cfg['split']['fold_index'])
split_pkl = Path(cfg['split']['predefined_split'])
with open(split_pkl, 'rb') as f:
    folds = pickle.load(f)
# 다양한 구조 지원: list of (train_idx, val_idx) or dict
if isinstance(folds, list):
    block = folds[fold_idx]
    if isinstance(block, (list, tuple)) and len(block) >= 2:
        val_idx = block[1]
    elif isinstance(block, dict):
        val_idx = block.get('val') or block.get('valid') or block.get('val_idx')
    else:
        raise ValueError('Unsupported folds list element structure')
elif isinstance(folds, dict):
    block = folds[fold_idx]
    val_idx = block.get('val') or block.get('valid') or block.get('val_idx')
else:
    raise ValueError('Unsupported folds structure')
val_idx = np.array(val_idx, dtype=int)

train_csv = Path(cfg['paths']['train_csv'])
df = pd.read_csv(train_csv)
val_df = df.iloc[val_idx].reset_index(drop=True)
num_classes = int(cfg['data']['num_classes'])

# 데이터/모델 로딩
import sys
SRC = ROOT / 'src'
if str(ROOT) not in sys.path: sys.path.append(str(ROOT))
from src.models.factory import create_model
from src.transforms.factory import create_transforms
from src.data.dataset import DocumentDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = create_model(cfg).to(device)
state = torch.load(run_dir/'checkpoints'/'best.pth', map_location=device)
state = state.get('ema_state') or state.get('swa_state') or state.get('model_state') or state
model.load_state_dict(state, strict=False)
model.eval()

tfm = create_transforms(cfg, is_train=False)
ds = DocumentDataset(val_df, Path(cfg['paths']['image_dir']), transforms=tfm, is_train=True)
loader = torch.utils.data.DataLoader(ds, batch_size=cfg['data']['loader'].get('batch_size', 8),
                                   shuffle=False, num_workers=cfg['data']['loader'].get('num_workers', 4),
                                   pin_memory=True)

logits_list, labels_list = [], []
with torch.no_grad():
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        out = model(x)
        logits_list.append(out.detach().cpu())
        labels_list.append(y.detach().cpu())
logits = torch.cat(logits_list, 0).numpy()
labels = torch.cat(labels_list, 0).numpy()
preds = logits.argmax(1)

# per-class correct/incorrect 막대그래프
correct = (preds == labels)
cnt_total = np.bincount(labels, minlength=num_classes)
cnt_correct = np.bincount(labels[correct], minlength=num_classes)
cnt_incorrect = cnt_total - cnt_correct
df_bar = pd.DataFrame({'class': range(num_classes), 'correct': cnt_correct, 'incorrect': cnt_incorrect})

stamp = time.strftime('%Y%m%d-%H%M%S')
out_dir = ROOT / 'reports' / 'summary' / stamp
out_dir.mkdir(parents=True, exist_ok=True)
ax = df_bar.set_index('class').plot(kind='bar', figsize=(10,4), title='Per-class Correct / Incorrect (Validation)')
plt.tight_layout(); plt.savefig(out_dir/'val_per_class_correct_incorrect.png', dpi=150); plt.close()

# 혼동행렬(정답 x 예측) 히트맵
cm = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(labels, preds): cm[t, p] += 1
plt.figure(figsize=(7,6))
sns.heatmap(cm, cmap='Blues', annot=False)
plt.title('Confusion (Val) True x Pred')
plt.xlabel('Pred'); plt.ylabel('True')
plt.tight_layout(); plt.savefig(out_dir/'val_confusion_true_pred.png', dpi=150); plt.close()

print('Saved:', out_dir)
